# Stage 04 — Calibration

Derives and applies gas-concentration calibration for the four WYO-platform gas
analyzers (**Picarro, Ultra460, Ultra321, Pico017**), using the tank/dilution
sequences logged in `raw/calibration/tank_details.txt`.

All the reusable machinery — manifest parsing, both fitting methods, the candidate-
comparison harness, applying coefficients, and the plotting primitives — lives in
**`src/calibration.py`** (imported as `cal` below). This notebook is the *narrated
applied pipeline*: campaign-specific configuration, the locked-in method per species,
and the checks that confirm it worked. No fitting or plotting logic is defined inline
here.

### One generalizable idea, two viable methods

For each species there are up to two candidate calibrations: **tank-anchored**
(regress against known tank/dilution concentrations) and **reference-instrument**
(cross-calibrate against another already-trusted instrument, either by continuous
ambient overlap or by matched plume peaks). Which candidates are actually *viable*
differs by species — CH4 has both viable; **C3H8 can only do tank** (Ultra321 is the
only instrument with that channel, so there's no reference partner to cross-cal
against); **C2H6 can only do reference** (the tank has just one certified point, far
below plume levels — not enough range to anchor a fit).

**This notebook applies whichever method is locked in** (`CAL_METHOD_LOCKED` below) —
it does not re-derive or compare candidates. The comparison evidence that justifies
each lock — including the discarded CH4 candidate, the C2H6 tank-coverage check, drift
QC across all three tank dates, and the Ultra321 C3H8-interference diagnostic — lives
in the companion notebook **`04_calibration_qc.ipynb`**, so this notebook stays a clean
description of what actually happens to the data. Re-run the QC notebook and edit
`CAL_METHOD_LOCKED` (with a matching code change to Section C/D) if new evidence should
change a lock.

### How to read this notebook

Sections run **A → H in order**; each builds on the last. Cells labeled **sanity
check** prove a step did what it claims (points collapsing onto a 1:1 line, corrected
traces tracking each other, residuals flat) via `cal.compare_candidate_coefs` — the
same function the QC notebook uses to compare *multiple* candidates, called here with
just the one locked-in candidate. The **Ultra321 C2H6 problem** (poor fit, strong C3H8
cross-interference) is flagged where it's applied and carried through to the "Open
items" recap in Section H — it is **not** resolved here; see the QC notebook for the
diagnostic plots behind it.

**Output** — `04_calibrated/` becomes the single complete Stage 04 directory:
- **Calibrated** (`Raw`+`Eng` for the four analyzers): gains `*_cal` columns and a
  `cal_coefs_ref` column pointing at `calibration_coefs.json`.
- **Passthrough** (Spectra/Spectralite + GPS/Anem/Sprinter/LGR): copied unchanged, no
  `cal_coefs_ref` column — that absence is how you tell passthrough from calibrated.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import importlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, str(Path().resolve().parent))
from paths import (STAGE_02_DIR, STAGE_03_DIR, STAGE_04_DIR,
                   TANK_DETAILS_PATH, REPO_ROOT)
from src import calibration as cal
from src.provenance import check_clean, upstream_ref
importlib.reload(cal)   # pick up edits to src/calibration.py without restarting the kernel
from src.align import load_aligned_series

print('Imports OK — calibration module reloaded')

In [ ]:
# ── Campaign-specific configuration ───────────────────────────────────────────
# Where each instrument's aligned gas data lives in Stage 03, and the color it wears
# in every plot below. Colors are a presentation choice for THIS instrument set, so
# they live here (the generic plotting functions take `colors` as a parameter).
INSTRUMENTS = {
    'Picarro':  {'dir': 'WYO_picarro',        'subdir': ''},
    'Ultra460': {'dir': 'WYO_aerisultra460',  'subdir': 'Raw'},
    'Ultra321': {'dir': 'LANL_aerisultra321', 'subdir': 'Raw'},
    'Pico017':  {'dir': 'LANL_aerispico017',  'subdir': 'Raw'},
}
INST_COLORS = {
    'Picarro':  '#1f77b4',   # blue
    'Ultra460': '#ff7f0e',   # orange
    'Ultra321': '#2ca02c',   # green
    'Pico017':  '#d62728',   # red
}
CAL_DATE_CANONICAL = '20260212'   # only event spanning the full 0-57 ppm dilution ladder

def load_full_series(inst, col):
    '''Concatenated Series for one instrument/column from all good Stage 03 aligned files.'''
    cfg = INSTRUMENTS[inst]
    return load_aligned_series(STAGE_03_DIR, cfg['dir'], cfg['subdir'], col)

# Which raw_stem sessions ran on the WYO platform, straight from Stage 02's routing
# manifest — loaded ONCE here so the C2H6 ambient-date restriction below and the QC
# notebook's own copy share this single source instead of each re-deriving it.
with open(STAGE_02_DIR / 'routing_manifest.json') as _f:
    ROUTING_MANIFEST = json.load(_f)
WYO_DATES = cal.dates_for_platform(ROUTING_MANIFEST, 'WYO')   # any instrument, any WYO date

# ── What a calibrated column MEANS (the product contract) ───────────────────────
# Stage 04 emits two suffixes, and the difference between them is the whole point:
#
#   *_cal   TRACEABLE. Derived against the certified tank/dilution ladder and nothing
#           else. CH4 (4 instruments) and C3H8 (Ultra321). A `*_cal` value is a
#           measurement referred to a certified standard.
#
#   *_xcal  TRANSFERRED. Harmonized to another INSTRUMENT because no certified anchor
#           spans the needed range. C2H6 (3 instruments, vs Ultra460). Internally
#           consistent across instruments; NOT traceable, and not an absolute
#           measurement. Ultra460 -- the reference -- carries the identity transform so
#           all three instruments expose the same column on the same footing.
#
# Anything requiring a campaign-specific modelling choice on top of these (the CH4
# baseline anchor below, drift correction, interference deconvolution) is recorded in
# calibration_coefs.json but NOT applied: that is analysis, and it lives downstream in
# mobile-hydrocarbon-analysis. This split is what keeps Stage 04 an ETL stage.
COL_SUFFIX_TRACEABLE   = '_cal'
COL_SUFFIX_TRANSFERRED = '_xcal'

# Per-instrument C2H6 source column + unit scaling into ppb. Ultra321 reports ppm;
# Ultra460 and Pico017 already report ppb. Ultra460 is the reference and gets the
# identity transform, so it needs an entry here too.
C2H6_INPUT = {
    'Ultra460': ('C2H6_ppb', 1.0),
    'Pico017':  ('C2H6_ppb', 1.0),
    'Ultra321': ('C2H6_ppm', 1000.0),
}

# ── Locked-in calibration method per species ─────────────────────────────────────
# Decided by comparing candidates in 04_calibration_qc.ipynb; this is the durable,
# human-reviewed choice actually applied below — NOT auto-inferred from the QC
# notebook at runtime. Editing this requires editing Section C/D's code to match (the
# asserts below catch a mismatch rather than silently applying the wrong method).
# NOTE on CH4: the method below is the SPAN method. Two instruments additionally have
# their baseline re-anchored -- see CAL_BASELINE_ANCHOR further down. That is an offset
# adjustment on top of this fit, not a different method, so 'tank' remains correct here.
CAL_METHOD_LOCKED = {
    'CH4':  'tank',       # both candidates viable (QC notebook §C): tank RMS-at-tank-points
                          # 0.08-0.27 ppm vs Picarro cross-cal's out-of-sample 0.19-0.50 ppm --
                          # tank chosen so the applied output stays trustworthy at PLUME
                          # concentrations (this is a plume-detection campaign), not just
                          # ambient, where the cross-cal is actually tighter. Locked 2026-07-07.
    'C3H8': 'tank',       # forced: only Ultra321 measures C3H8, no reference candidate exists.
    'C2H6': 'reference',  # forced: tank has 1 certified pt (NOAA, 1.63 ppb), far below the
                          # plume range that must be covered — see QC notebook §E
                          # (cal.assess_tank_coverage). Ultra460 is the reference.
}
assert CAL_METHOD_LOCKED['CH4'] == 'tank', 'Section C below only implements the tank path'
assert CAL_METHOD_LOCKED['C3H8'] == 'tank', 'Section C below only implements the tank path'
assert CAL_METHOD_LOCKED['C2H6'] == 'reference', 'Section D below only implements the reference path'

# ── Corrections deliberately NOT applied ─────────────────────────────────────────
# A method lock says HOW a species is calibrated; this says which individual
# (gas, instrument) corrections are withheld from the output despite being
# computable. The fit is still COMPUTED in the section below — the evidence behind
# a drop stays visible in this notebook — but the correction is excluded from
# calibration_coefs.json and never applied, so the instrument simply has no
# calibrated column for that gas.
#
# Currently empty. ('C2H6', 'Ultra321') was dropped 2026-08-26 and REINSTATED
# 2026-08-27: its C3H8 cross-talk is itself a deliverable — characterising when a
# C2H6 retrieval degrades in the presence of propane/methane needs the calibrated
# column, not its absence — so it now ships with a CAL_CAVEATS entry instead.
CAL_DROPPED = {}

# ── Corrections applied WITH a documented reliability caveat ─────────────────────
# Same (gas, instrument) key shape as CAL_DROPPED, opposite disposition: the
# correction IS saved and applied, but carries a `caveat` string into its record in
# calibration_coefs.json so neither downstream code nor a reader can mistake it for
# an unqualified result. Presence of a 'caveat' field is the machine-readable flag.
# Deliberately quotes NO fit statistics — the record's own `r2` is written at run
# time and the interference diagnostics live in the QC notebook, so this text cannot
# go stale when the alignment changes.
CAL_CAVEATS = {
    ('C2H6', 'Ultra321'): (
        'RETAINED FOR DIAGNOSTIC USE, NOT AS A QUANTITATIVE C2H6 MEASUREMENT. This '
        'channel suffers C3H8 spectral cross-talk: at the matched plume peaks its fit '
        'residual correlates strongly and positively with Ultra321 C3H8, and its span '
        'fit is far looser than Pico017 (compare the r2 fields of the two C2H6 '
        'corrections in this file; the interference diagnostic itself is in '
        '04_calibration_qc.ipynb section E). The ambient-median anchor makes the '
        'baseline match Ultra460 by construction, so the BASELINE looks right while '
        'individual peak magnitudes are not reliable -- no anchor choice fixes a '
        'spectral interference. It is shipped anyway, as C2H6_ppb_xcal, because '
        'characterising when a C2H6 retrieval degrades in the presence of '
        'propane/methane requires the corrected series: treat it as evidence about the '
        'instrument, and propagate the uncertainty explicitly before using it as a '
        'measurement. Dropped 2026-08-26, reinstated on this basis 2026-08-27.'),
}

# ── Baseline re-anchoring: COMPUTED AND RECORDED, NOT APPLIED ────────────────────
# A third disposition alongside CAL_DROPPED/CAL_CAVEATS, same (gas, instrument) key
# shape. For each listed correction an ALTERNATIVE intercept is derived -- tank slope
# kept bit-for-bit, offset moved so the ambient baseline matches co-located,
# tank-corrected Picarro at CAL_ANCHOR_Q -- and written into calibration_coefs.json as a
# `baseline_anchor` block carrying "applied": false.
#
# WHY IT IS NOT APPLIED (product decision 2026-09-01). Per the contract at the top of
# this cell, `*_cal` means traceable-to-the-certified-ladder. This anchor is a
# harmonization to another instrument -- a different kind of claim -- so it cannot share
# that column. Nothing is lost by withholding it: it is a pure constant offset, so
# downstream applies it in one line from the recorded `intercept_if_applied`, and
# enhancement (dCH4) work is identical either way because a constant cancels. Set
# CAL_BASELINE_ANCHOR_APPLY = True to fold it back into *_cal instead.
#
# WHY THE OFFSET EXISTS IN THE FIRST PLACE: the tank fit is an unweighted OLS over 0-57 ppm. Ambient sits at ~2 ppm, the
# bottom 3.5% of that range, so a constant offset down there costs the fit essentially
# nothing against matching the 25 and 57 ppm points -- Ultra460's tank correction even
# changes sign at ~5 ppm. Measured against tank-corrected Picarro, 85-98% of each
# instrument's ambient mean-square error is that constant offset, and the residual
# SCATTER is identical with or without this adjustment (e.g. Ultra460 0.0308 ppm either
# way). So this moves the zero and changes nothing else; DeltaCH4 is unaffected entirely.
#
# EVIDENCE (04_calibration_qc.ipynb SS-C, and a held-out check fitting the anchor on one
# set of WYO dates and scoring it on the other): held-out ambient RMS improves from
# ~0.13 to ~0.03 ppm for Ultra460 and ~0.12 to ~0.04 for Ultra321. A relative-error
# WEIGHTED tank fit was also tried, to fix the low end without any reference instrument,
# and it FAILED -- ambient bias got worse, not better. The tank's own 0 and 2 ppm
# dilution points do not agree with what co-located ambient says, so there is no
# tank-only fix available.
#
# WHY PICO017 IS DELIBERATELY ABSENT: Pico017 is co-located with Picarro only on the
# 8 WYO days (Feb 5-12); 17.9% of its rows are MML sessions spanning Jan 19 - Mar 10.
# Comparing Pico017 against Ultra321 -- co-located on every MML day -- shows their
# relative CH4 baseline moving over 0.63 ppm across the campaign (+0.89 on Jan 20 down
# to +0.26 on Mar 10) while holding to 0.09 ppm inside the WYO window. An anchor derived
# from that one week is therefore not safely extrapolated to the rest of Pico017's
# record, and the drift is larger than anything this adjustment would fix. Pico017 keeps
# the plain tank fit; correcting the campaign drift is post-analysis, not ETL.
# Pad applied around every tank window when carving out an 'ambient only' population.
# Shared by the CH4 baseline anchor (Section C) and the C2H6 ambient anchor (Section D)
# so the two cannot silently diverge.
CAL_WINDOW_PAD_MIN = 5
CAL_ANCHOR_Q = 0.5      # median: robust to plumes, and the same convention C2H6 uses
CAL_BASELINE_ANCHOR_APPLY = False   # False => recorded in the coefs file, absent from *_cal
CAL_BASELINE_ANCHOR = {
    ('CH4', 'Ultra460'): 'Tank span retained; baseline matched to co-located Picarro.',
    ('CH4', 'Ultra321'): 'Tank span retained; baseline matched to co-located Picarro.',
}

# ── Per-species results container ─────────────────────────────────────────────────
# Replaces a proliferation of near-duplicate globals (one set per species) with one
# dict: RESULTS[species]['raw'] (loaded series), ['candidates'][method] (coefficients
# for whichever method(s) this notebook fits), ['locked'] (the method name actually
# applied), ['applied'] (alias to candidates[locked] once fit -- what Section E saves).
RESULTS = {sp: {'raw': None, 'candidates': {}, 'locked': m} for sp, m in CAL_METHOD_LOCKED.items()}

print('Instruments:', list(INSTRUMENTS))
print('Canonical calibration date:', CAL_DATE_CANONICAL)
print('WYO co-deployment dates:', len(WYO_DATES))
print('Locked calibration method per species:', CAL_METHOD_LOCKED)
if CAL_DROPPED:
    print('Corrections DROPPED (computed but not saved/applied):',
          ', '.join(f'{g}/{i}' for g, i in sorted(CAL_DROPPED)))
if CAL_CAVEATS:
    print('Corrections CAVEATED (applied, flagged in the coefs file):',
          ', '.join(f'{g}/{i}' for g, i in sorted(CAL_CAVEATS)))
if CAL_BASELINE_ANCHOR:
    _disp = 'APPLIED to *_cal' if CAL_BASELINE_ANCHOR_APPLY else 'RECORDED ONLY, not applied'
    print(f'Baseline anchor at q={CAL_ANCHOR_Q} vs Picarro [{_disp}]:',
          ', '.join(f'{g}/{i}' for g, i in sorted(CAL_BASELINE_ANCHOR)))
print(f'Column contract: {COL_SUFFIX_TRACEABLE} = tank-traceable, '
      f'{COL_SUFFIX_TRANSFERRED} = cross-instrument transfer.')

---
## A — Parse the tank calibration manifest

`tank_details.txt` lists the certified concentration of each tank/dilution and the UTC
time windows during which each was delivered on each calibration date. `parse_tank_details`
returns `(TANK, WINDOWS_BY_DATE)`. All three tank dates' windows are kept (not just the
canonical one) — even though Section C only *fits* on Feb 12, ambient-population
restriction in Section D excludes tank gas from every date it was delivered.

In [ ]:
TANK, WINDOWS_BY_DATE = cal.parse_tank_details(TANK_DETAILS_PATH)

print('Tank standards:', list(TANK), '\n')
for date, wins in sorted(WINDOWS_BY_DATE.items()):
    tag = '   <- canonical (full dilution ladder)' if date == CAL_DATE_CANONICAL else ''
    print(f'{date}: {len(wins)} windows{tag}')
    print('   ', [w['tank_key'] for w in wins])

The certified concentrations, as a table. Note the asymmetry that shapes the rest of
the notebook: **CH4 and C3H8 span a full ladder** (zero → 57 ppm / 10 ppm across five
dilutions + NOAA), while **C2H6 has exactly one certified value** (NOAA, 1.63 ppb) plus
the implicit zero. That single low point is why C2H6 cannot use the tank method — see
the QC notebook's `assess_tank_coverage` table for the numeric version of this claim.

In [ ]:
tank_df = pd.DataFrame(TANK).T[['CH4_ppm', 'C3H8_ppm', 'C2H6_ppb']]
tank_df.index.name = 'tank_key'
tank_df

---
## B — Load Stage 03 aligned data

One concatenated Series per instrument per species, pulled straight from the good
(non-`bad`, non-`bad_timestamp`) Stage 03 output. Ultra321 reports C2H6 in ppm; it is
converted to ppb here so all three C2H6 series share units.

In [ ]:
RESULTS['CH4']['raw']  = {inst: load_full_series(inst, 'CH4_ppm') for inst in INSTRUMENTS}
RESULTS['C3H8']['raw'] = {'Ultra321': load_full_series('Ultra321', 'C3H8_ppm')}   # only Ultra321 has C3H8
RESULTS['C2H6']['raw'] = {
    'Ultra460': load_full_series('Ultra460', 'C2H6_ppb'),
    'Pico017':  load_full_series('Pico017', 'C2H6_ppb'),
    'Ultra321': load_full_series('Ultra321', 'C2H6_ppm') * 1000.0,     # ppm -> ppb
}

for inst, s in RESULTS['CH4']['raw'].items():
    print(f'{inst:10s} CH4   {len(s):>9,} pts   {s.index[0]} -> {s.index[-1]}')
print()
for inst, s in RESULTS['C2H6']['raw'].items():
    print(f'{inst:10s} C2H6  {len(s):>9,} pts')

**The "before" picture — a specific window at native resolution.** Rather than averaging
the whole campaign down to 10-minute means (which smears out exactly the plume structure
we care about), pick one time window and look at the raw 1 Hz data as stored — set
`ZOOM_START` / `ZOOM_END` below to any period you want to inspect. Section F re-plots this
*same window after* calibration, so the effect is directly comparable. Nothing here is
resampled, averaged, or interpolated.

In [ ]:
# ── Set your inspection window here ─────────────────────────────────────────────
# Any UTC period you want to inspect at native resolution — this window is reused in
# Section F's "did it help" check, so picking one custom range lets you watch the same
# period get calibrated end to end.
# ⚠️ The default below (Feb 5) is one of the WORST days for CH4 baseline agreement — the
# residual after re-anchoring is ~2-6x larger there than on Feb 7-12 (see Section H's
# per-date table). It is kept as the default deliberately, so this check shows a hard case
# rather than a flattering one. Move it to Feb 9-12 to see the typical case.
ZOOM_START = pd.Timestamp('2026-02-05 20:00', tz='UTC')   # e.g. pd.Timestamp('2026-02-11 20:30', tz='UTC')
ZOOM_END   = ZOOM_START + pd.Timedelta('180min')          # e.g. ZOOM_START + pd.Timedelta('40min')

print('Inspection window:', ZOOM_START, '->', ZOOM_END)

fig = cal.plot_timeseries_panels(
    panels=[('CH4 — raw, all instruments', 'CH4 (ppm)', RESULTS['CH4']['raw']),
            ('C2H6 — raw', 'C2H6 (ppb)', RESULTS['C2H6']['raw']),
            ('C3H8 — Ultra321 raw', 'C3H8 (ppm)', RESULTS['C3H8']['raw'])],
    colors=INST_COLORS, t0=ZOOM_START, t1=ZOOM_END,
    title=f'Raw Stage 03 data at native resolution — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC')
fig.show()

---
## C — CH4 & C3H8 tank calibration (locked method: `tank`)

A single multi-point OLS fit per instrument per date — **no** piecewise low/high split.
A single fit already reaches R² > 0.9997 for every instrument on Feb 12, and the small
residual wiggle that once motivated a piecewise split appears *identically* in Picarro
(the reference-grade, presumptively-linear instrument), which means it reflects tiny
imprecision in the dilution manifold's delivered concentrations, not instrument
nonlinearity. Splitting would just overfit that artifact.

**Why tank over reference-cross-cal for CH4** (the QC notebook's full comparison): the
tank spans CH4's full working range (0–57 ppm) so it stays accurate at plume
concentrations, which is what this campaign is measuring; a reference cross-cal against
Picarro's own ambient data is tighter at *ambient* levels but degrades noticeably at
high concentrations, since its training population is >99.9% ambient (<5 ppm).

First, **see what is being averaged**: the Feb 12 tank sequence with each delivery
window shaded and labeled. The flat plateaus inside the shaded spans are what
`window_stats` bins into a single mean per tank.

In [ ]:
feb12_windows = WINDOWS_BY_DATE[CAL_DATE_CANONICAL]
fig = cal.plot_timeseries_with_windows(
    RESULTS['CH4']['raw'], feb12_windows, INST_COLORS,
    title='Feb 12 CH4 during the tank sequence — shaded spans are the fit windows',
    y_title='CH4 (ppm)')
fig.show()

In [ ]:
fig = cal.plot_timeseries_with_windows(
    RESULTS['C3H8']['raw'], feb12_windows, INST_COLORS,
    title='Feb 12 C3H8 (Ultra321) during the tank sequence',
    y_title='C3H8 (ppm)')
fig.show()

Now the fits. `window_stats` computes each instrument's mean over the Feb 12 windows;
`fit_species` regresses those means against the known tank concentration.

In [ ]:
CH4_STATS_CANONICAL  = cal.window_stats(RESULTS['CH4']['raw'], feb12_windows)
C3H8_STATS_CANONICAL = cal.window_stats(RESULTS['C3H8']['raw'], feb12_windows)

RESULTS['CH4']['candidates']['tank']  = {inst: cal.fit_species(CH4_STATS_CANONICAL, TANK, inst, 'CH4_ppm')
                                         for inst in INSTRUMENTS}
RESULTS['C3H8']['candidates']['tank'] = {'Ultra321': cal.fit_species(C3H8_STATS_CANONICAL, TANK, 'Ultra321', 'C3H8_ppm')}

RESULTS['CH4']['applied']  = RESULTS['CH4']['candidates'][RESULTS['CH4']['locked']]
RESULTS['C3H8']['applied'] = RESULTS['C3H8']['candidates'][RESULTS['C3H8']['locked']]
print('Feb 12 tank fits computed. CH4 baseline re-anchoring is applied in the next cell.')

**Baseline re-anchoring (CH4 only) — computed here, recorded in the coefficients
file, and deliberately NOT applied.** The tank fit above is an unweighted OLS spanning
0–57 ppm, so it is pinned by its high-concentration points and can carry a constant
offset at ambient (~2 ppm) essentially for free. For the instruments listed in
`CAL_BASELINE_ANCHOR`, `cal.reanchor_intercept` derives an alternative intercept that
keeps the fitted **slope untouched** and moves only the offset, so the ambient median
would match co-located, tank-corrected Picarro.

**Why it is not applied.** A `*_cal` column in this dataset means *traceable to the
certified ladder*. This adjustment is a harmonization to another **instrument**, which
is a different kind of claim, so folding it into the same column is exactly the
"half-calibration, half-analysis" blend Stage 04 is meant to avoid. It costs nothing to
withhold: the adjustment is a pure constant, so anyone wanting inter-instrument
agreement on the WYO days applies it in one line from `baseline_anchor.intercept_if_applied`
in `calibration_coefs.json`, and **ΔCH4 is unaffected either way** because a constant
cancels. The numbers below are printed so the size of the effect is visible on the page.

> **Pico017 is deliberately not in the list.** Its baseline drifts by more than 0.6 ppm
> relative to Ultra321 across the campaign while holding to 0.09 ppm inside the WYO week,
> so a one-week anchor cannot be extrapolated to the 18% of its record collected on MML
> days between January and March. See Section H.


In [ ]:
# Ambient-only: WYO co-deployment dates, tank windows excluded with the shared pad.
_pic_amb = cal.restrict_series(RESULTS['CH4']['raw']['Picarro'], WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
_pic_cal = cal.apply_linear(_pic_amb, RESULTS['CH4']['applied']['Picarro'])

CH4_ANCHOR_PAIRS = {}
CH4_ANCHOR_COEFS = {}   # alternative coefficients: recorded in Section E, applied only
                        # if CAL_BASELINE_ANCHOR_APPLY (default False)
for (_gas, _inst), _reason in sorted(CAL_BASELINE_ANCHOR.items()):
    if _gas != 'CH4':
        continue
    _tgt = cal.restrict_series(RESULTS['CH4']['raw'][_inst], WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
    _m   = cal.pair_series_nearest(_pic_cal, _tgt, tolerance_s=1)
    CH4_ANCHOR_PAIRS[_inst] = _m
    _before = RESULTS['CH4']['applied'][_inst]
    _after  = cal.reanchor_intercept(_before, _m['ref'].values, _m['target'].values, q=CAL_ANCHOR_Q)
    CH4_ANCHOR_COEFS[_inst] = _after
    if CAL_BASELINE_ANCHOR_APPLY:
        RESULTS['CH4']['applied'][_inst] = _after
    _bias_b = float(np.median(cal.apply_linear(_m['target'], _before) - _m['ref']))
    _bias_a = float(np.median(cal.apply_linear(_m['target'], _after)  - _m['ref']))
    print(f"{_inst}: slope {_after['slope']:.6f} UNCHANGED   "
          f"intercept {_after['intercept_prior']:+.6f} -> {_after['intercept']:+.6f} "
          f"({'APPLIED' if CAL_BASELINE_ANCHOR_APPLY else 'recorded only'})")
    print(f"    anchored at q={_after['anchor_q']} on {_after['n_anchor']:,} paired ambient pts   "
          f"median bias vs Picarro {_bias_b:+.4f} -> {_bias_a:+.4f} ppm")

for _inst in INSTRUMENTS:
    if ('CH4', _inst) not in CAL_BASELINE_ANCHOR:
        print(f'{_inst}: plain tank fit, no baseline anchor computed.')

if not CAL_BASELINE_ANCHOR_APPLY:
    print('\n*** CH4_ppm_cal keeps the PLAIN TANK FIT for every instrument. The anchored'
          '\n    intercepts above are saved to calibration_coefs.json as'
          '\n    baseline_anchor.intercept_if_applied (applied=false) for downstream use.')


In [ ]:
rows = []
for inst, c in RESULTS['CH4']['applied'].items():
    if c: rows.append({'gas': 'CH4', 'instrument': inst, **c})
for inst, c in RESULTS['C3H8']['applied'].items():
    if c: rows.append({'gas': 'C3H8', 'instrument': inst, **c})
pd.DataFrame(rows).set_index(['gas', 'instrument']).round(5)

**The 1:1 view.** Points are window means; **error bars are ±1σ of the in-window noise**
(how steadily each instrument held during that tank — usually smaller than the marker).
The solid line is each instrument's fit; the dashed line is `y = x`. Distance from the
dashed line *is* the calibration error the fit corrects. A perfect instrument would already
sit on the dashed line.

In [ ]:
def scatter_xy(stats_df, tank, insts, species_key):
    '''Build {inst: x/y/err arrays} for plot_calibration_scatter (err = 1σ in-window noise).'''
    xt = stats_df['tank_key'].map(lambda k: tank.get(k, {}).get(species_key)).astype(float)
    x, y, err = {}, {}, {}
    for inst in insts:
        if f'{inst}_mean' in stats_df.columns:
            x[inst] = xt.values
            y[inst] = stats_df[f'{inst}_mean'].astype(float).values
            if f'{inst}_std' in stats_df.columns:
                err[inst] = stats_df[f'{inst}_std'].astype(float).values
    return x, y, err

xg, yg, eg = scatter_xy(CH4_STATS_CANONICAL, TANK, INSTRUMENTS, 'CH4_ppm')
fig = cal.plot_calibration_scatter(xg, yg, RESULTS['CH4']['applied'], INST_COLORS,
        x_title='Tank CH4 (ppm)', y_title='Instrument CH4 (ppm)',
        title='CH4 calibration (Feb 12) — points ±1σ in-window noise, per-instrument fit, 1:1 line',
        yerr_by_group=eg)
fig.show()

In [ ]:
xg, yg, eg = scatter_xy(C3H8_STATS_CANONICAL, TANK, ['Ultra321'], 'C3H8_ppm')
fig = cal.plot_calibration_scatter(xg, yg, RESULTS['C3H8']['applied'], INST_COLORS,
        x_title='Tank C3H8 (ppm)', y_title='Ultra321 C3H8 (ppm)',
        title='C3H8 calibration (Feb 12) — Ultra321, points ±1σ in-window noise',
        yerr_by_group=eg)
fig.show()

**Sanity check**, via `cal.compare_candidate_coefs` — the same function the QC notebook
uses to compare *multiple* candidates, called here with just the one locked-in `tank`
candidate. Apply the fit back to its own window means: if the fit is good, corrected
points collapse onto the dashed 1:1 line and the residual columns below stay small
(units: ppm/ppm).

In [ ]:
for inst in INSTRUMENTS:
    coef = RESULTS['CH4']['applied'].get(inst)
    if coef is None:
        continue
    cmp = cal.compare_candidate_coefs({'tank': coef}, CH4_STATS_CANONICAL, TANK, 'CH4_ppm', inst)
    cmp.insert(0, 'instrument', inst)
    print(cmp.to_string())

coef = RESULTS['C3H8']['applied']['Ultra321']
cmp = cal.compare_candidate_coefs({'tank': coef}, C3H8_STATS_CANONICAL, TANK, 'C3H8_ppm', 'Ultra321')
cmp.insert(0, 'instrument', 'Ultra321 (C3H8)')
print(cmp.to_string())

> **Why no CH4 cross-cal or drift-QC content here.** A Picarro-referenced ambient
> cross-cal candidate for CH4, and a 3-tank-date drift check, are both computed in
> `04_calibration_qc.ipynb` — that's the evidence behind locking `tank` in above, and
> behind the baseline re-anchoring applied to Ultra460 and Ultra321. This notebook
> applies the lock; it doesn't re-derive the case for it.
>
> **A limit of that drift check worth knowing:** all three tank events (Feb 3, 6, 12)
> fall inside the WYO window, so it can only ever demonstrate stability across Feb 3–12.
> It has no power over the January and March MML sessions. Section H records what a
> co-located Pico017-vs-Ultra321 comparison shows there.

---
## D — C2H6 reference-instrument calibration (locked method: `reference`)  ⚠️ FLAGGED — lower confidence

Not tank-anchored — the tank has only one certified C2H6 point (NOAA, 1.63 ppb), far
below plume levels (see the QC notebook's `assess_tank_coverage` table for the numeric
case). Instead **Ultra460 is treated as the C2H6 reference**; Pico017 and Ultra321 are
harmonized to it by matching plume-peak magnitudes, over every WYO co-deployment day
(from `routing_manifest.json`), with the tank windows excluded.

> **This is why C2H6 emits `*_xcal`, not `*_cal`.** Nothing in this section is referred
> to a certified standard, so calling the result "calibrated" would overstate it. Every
> coefficient here carries `traceability: "none_cross_instrument_transfer"` and
> `confidence: "low"`. **Ultra460 is included with the identity transform** — its
> `C2H6_ppb_xcal` equals its raw reading — so all three instruments expose the same
> column on the same footing, instead of the product being "one instrument raw, two
> transformed to match it". The transfer makes them mutually consistent; it does not make
> any of them absolute.

> **Ultra321 C2H6 is calibrated, but caveated** (`CAL_CAVEATS` in the config cell). Its
> C3H8 cross-talk makes it unreliable as a quantitative C2H6 measurement, and it was
> briefly dropped for exactly that reason on 2026-08-26 — but the corrected series is
> what is needed to characterise *when* a C2H6 retrieval degrades in the presence of
> propane, so it ships with a `caveat` field in `calibration_coefs.json` rather than
> being withheld. All three instruments get a `C2H6_ppb_xcal` column; Ultra460's is the
> identity, and it remains the uncalibrated reference. See the box at the end of this
> section.

In [ ]:
# cal.restrict_series (generic: date filter + optional tank-window exclusion) keeps
# tank gas out of what should be an ambient-only population.
U460_AMB = cal.restrict_series(RESULTS['C2H6']['raw']['Ultra460'], WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
PICO_AMB = cal.restrict_series(RESULTS['C2H6']['raw']['Pico017'],  WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
U321_AMB = cal.restrict_series(RESULTS['C2H6']['raw']['Ultra321'], WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
print(f'WYO ambient dates used: {len(WYO_DATES)}   Ultra460 ambient C2H6 points: {len(U460_AMB):,}')

`find_peak_matches` locates plume peaks in the Ultra460 reference (per day) and, at
each peak time, grabs the local max of Pico017 and Ultra321 within a ±10 s window. (The
QC notebook additionally pulls CH4/C3H8 at each peak, for the interference diagnostic
behind the Ultra321 flag below — not needed to compute the fit itself.)

In [ ]:
PEAK_HEIGHT_PPB     = 50.0
PEAK_PROMINENCE_PPB = 15.0
PEAK_MIN_DISTANCE_S = 30
PEAK_MATCH_WINDOW_S = 10

peaks_df = cal.find_peak_matches(
    U460_AMB, {'Pico017': PICO_AMB, 'Ultra321': U321_AMB},
    height=PEAK_HEIGHT_PPB, prominence=PEAK_PROMINENCE_PPB,
    min_distance_s=PEAK_MIN_DISTANCE_S, window_s=PEAK_MATCH_WINDOW_S)
print(f'Plume peaks found: {len(peaks_df)}')
peaks_df.groupby('date').size().rename('n_peaks').to_frame()

**The zero+span fit.** Two anchors: the **baseline** is fixed first — matched directly to
Ultra460's own ambient level, not the tank — and the **plume peaks** fix the gain/span
against Ultra460 second. This is `cal.fit_reference_cal(anchor='pinned')` under the hood
(`cal.calibrate_and_check_reference` runs it plus the standard checks in one call).

> **Why match Ultra460's baseline instead of the tank's absolute zero.** An earlier
> version of this fit anchored to the certified tank N2-zero instead. That's the more
> "physically true" anchor, but it left Pico017's corrected baseline sitting ~5–6 ppb
> *below* Ultra460's, because the tank says Pico017's true background C2H6 ≈ 0 while
> Ultra460 reads ~+5–7 ambient — the two instruments genuinely disagree at baseline. Since
> the entire point of this section is cross-instrument agreement with Ultra460 (not an
> absolute-truth measurement), **the baseline anchor is set to match Ultra460's own
> ambient level first**, by construction, and the gain is fit on top of that. This means
> the corrected reading is not independently traceable to the certified zero — a
> deliberate trade, made explicit here and in the saved coefficients' `note` field.

In [ ]:
Z_REF, Z_REF_SPREAD, _ = cal.ambient_baseline_stats(U460_AMB, q=0.5)
C2H6_BASELINE = {'Ultra460': (Z_REF, Z_REF_SPREAD)}
RESULTS['C2H6']['candidates']['reference'] = {}
for inst, amb in [('Pico017', PICO_AMB), ('Ultra321', U321_AMB)]:
    sub = peaks_df.dropna(subset=['ref', inst])
    z_tgt, z_tgt_spread, _ = cal.ambient_baseline_stats(amb, q=0.5)
    C2H6_BASELINE[inst] = (z_tgt, z_tgt_spread)
    c = cal.calibrate_and_check_reference(
        sub['ref'].values, sub[inst].values, anchor='pinned', z_ref=Z_REF, z_tgt=z_tgt,
        colors=INST_COLORS, x_label='Ultra460 C2H6 peak (ppb)',
        y_label='Instrument C2H6 peak (ppb)', target_label=inst,
        title_prefix='C2H6 zero+span vs Ultra460 (FLAGGED) — ')
    c['z_ref_std'], c['z_tgt_std'] = Z_REF_SPREAD, z_tgt_spread
    RESULTS['C2H6']['candidates']['reference'][inst] = c
    print(f"{inst}:  ambient baseline {z_tgt:+7.1f}±{z_tgt_spread:.1f} -> {Z_REF:+.1f}±{Z_REF_SPREAD:.1f} ppb   "
          f"gain={c['gain']:.4f}  slope={c['slope']:.4f}  intercept={c['intercept']:+.2f}  "
          f"R2(span)={c['r2']:.4f}  n={c['n']}")

# Ultra460 is the reference, so its own transform is the IDENTITY -- but it is recorded
# and emitted like any other correction rather than left as a bare passthrough. That is
# what puts all three instruments on the same footing: every one of them exposes a
# C2H6_ppb_xcal column, and Ultra460's simply happens to equal its raw reading. Without
# this, the product would silently be "one instrument raw, two transformed to match it",
# which is precisely the asymmetry the _xcal contract exists to remove.
RESULTS['C2H6']['candidates']['reference']['Ultra460'] = {
    'slope': 1.0, 'intercept': 0.0, 'gain': 1.0,
    'z_ref': Z_REF, 'z_tgt': Z_REF, 'z_ref_std': Z_REF_SPREAD, 'z_tgt_std': Z_REF_SPREAD,
    'r2': None, 'n': None, 'identity': True,
}
print(f"Ultra460: REFERENCE -- identity transform (slope=1, intercept=0), "
      f"ambient baseline {Z_REF:+.1f}+/-{Z_REF_SPREAD:.1f} ppb, uncalibrated by construction")

# The locked method's candidates, MINUS anything listed in CAL_DROPPED (config cell).
# Every instrument above was still fit, so the diagnostics below remain complete; only
# what reaches Section E's calibration_coefs.json and Section F's apply is filtered.
_c2h6_locked = RESULTS['C2H6']['candidates'][RESULTS['C2H6']['locked']]
RESULTS['C2H6']['applied'] = {inst: c for inst, c in _c2h6_locked.items()
                              if ('C2H6', inst) not in CAL_DROPPED}
for _inst in sorted(set(_c2h6_locked) - set(RESULTS['C2H6']['applied'])):
    print(f'\n  *** DROPPED -- C2H6/{_inst}: fit computed above, NOT saved and NOT applied.')
    print(f'      {CAL_DROPPED[("C2H6", _inst)]}')
for _inst in sorted(RESULTS['C2H6']['applied']):
    if ('C2H6', _inst) in CAL_CAVEATS:
        print(f'\n  *** CAVEATED -- C2H6/{_inst}: applied and saved, WITH a documented caveat.')
        print(f'      {CAL_CAVEATS[("C2H6", _inst)]}')

**Timeseries version, same custom window as everywhere else.** Ultra460 (reference)
plotted against each target's raw and corrected trace. Since the anchor is now the
ambient baseline, the two traces should sit on Ultra460's *baseline* here even outside a
plume — that's the direct visual proof the new anchor does what it's supposed to.

In [ ]:
# Driven off ['applied'], so a correction dropped in the config cell simply drops out
# of this check too rather than raising a KeyError. Ultra460's identity transform is
# skipped -- it is drawn separately as the reference trace, and plotting it twice would
# just overlay a line on itself.
_c2h6_begin = {inst: (RESULTS['C2H6']['raw'][inst],
                      cal.apply_linear(RESULTS['C2H6']['raw'][inst], c))
               for inst, c in RESULTS['C2H6']['applied'].items()
               if not c.get('identity')}
_cmp_colors = {'Ultra460 (reference)': INST_COLORS['Ultra460'],
               'Pico017': INST_COLORS['Pico017'], 'Ultra321': INST_COLORS['Ultra321']}
fig = cal.plot_raw_corrected_vs_reference(
    RESULTS['C2H6']['raw']['Ultra460'], 'Ultra460 (reference)',
    _c2h6_begin,
    _cmp_colors, ZOOM_START, ZOOM_END,
    title=f'C2H6 — beginning-of-calibration check — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC',
    y_title='C2H6 (ppb)')
fig.show()

> ### ⚠️ Ultra321 C2H6 — SHIPPED AS `_xcal`, BUT NOT A QUANTITATIVE MEASUREMENT
>
> The ambient-baseline anchor fixes Ultra321's *baseline* to match Ultra460 by
> construction — so the old "reads negative most of the campaign" symptom is gone at
> baseline. Its **peaks remain unreliable**: the span fit is far looser than Pico017's,
> and the fit residual correlates strongly and positively with C3H8 at the matched peaks
> (diagnostic in the QC notebook, §E) — the signature of **C3H8 leaking into the C2H6
> retrieval**. No anchor choice fixes a spectral interference.
>
> **Verified on the corrected timeline.** Re-running the QC notebook against the
> logger-host-clock Stage 03 output *reduced* the interference correlation but left the
> span R² essentially unchanged — some of the old correlation was timing smear, the
> defect underneath is structural. The binned-ambient alternative candidate was worse
> still: its ambient bins are flat across the entire Ultra460 4–17 ppb range, i.e. the
> channel barely responds to real ambient C2H6 at all.
>
> **Decision (2026-08-27): retain and ship it, with a caveat rather than a drop.** It was
> dropped on 2026-08-26 and reinstated the next day — the failure of this channel in the
> presence of propane/methane is itself a result to be characterised downstream, and that
> work needs the calibrated series, not its absence. It is listed in `CAL_CAVEATS` (config
> cell), so its record in `calibration_coefs.json` carries a `caveat` field alongside
> `confidence: "low"` and `traceability: "none_cross_instrument_transfer"`. **Downstream: treat it as evidence about the instrument, not as a
> C2H6 measurement, and propagate the uncertainty explicitly.**
>
> Live numbers for both C2H6 instruments are printed by the fit cell above and by the QC
> notebook's §E interference diagnostic — deliberately not restated here, so this box
> cannot go stale when the alignment changes.


---
## E — Assemble and save `calibration_coefs.json`

Each correction records the formula inputs (`slope`, `intercept`, `scale_in`), the
`col_in` → `col_out` mapping, and `method` / `traceability` / `confidence` tags. The
apply formula is `calibrated = (measured * scale_in - intercept) / slope`.

**`traceability` is the field that matters for interpretation**, and it takes exactly
two values:

| value | column | meaning |
|---|---|---|
| `certified_tank_ladder` | `*_cal` | referred to the certified dilution ladder — CH4, C3H8 |
| `none_cross_instrument_transfer` | `*_xcal` | harmonized to Ultra460 — C2H6, internally consistent but not absolute |

Anything needing a campaign-specific modelling choice on top of that is written with
`"applied": false` and left out of the data columns — currently just the CH4
`baseline_anchor`. The `metadata.column_contract` block states all of this inside the
file itself, so a downstream reader never has to come back here to interpret a column.

In [ ]:
check_clean(REPO_ROOT, context='Stage 04')
git_hash, git_dirty = cal.git_info(REPO_ROOT)

corrections = []
for inst, c in RESULTS['CH4']['applied'].items():
    if c is None:
        continue
    _anchor = CH4_ANCHOR_COEFS.get(inst)
    _rec = {'gas': 'CH4', 'instrument': inst,
            'col_in': 'CH4_ppm', 'col_out': 'CH4_ppm' + COL_SUFFIX_TRACEABLE,
            'scale_in': 1.0,
            'method': 'tank_multipoint_ambient_anchored' if CAL_BASELINE_ANCHOR_APPLY and _anchor
                      else 'tank_multipoint',
            'traceability': 'certified_tank_ladder', 'confidence': 'high',
            'cal_date': CAL_DATE_CANONICAL, 'slope': c['slope'], 'intercept': c['intercept'],
            'r2': c['r2'], 'n_points': c['n'],
            'method_selection_note': ('CAL_METHOD_LOCKED["CH4"]="tank" -- see '
                     '04_calibration_qc.ipynb for the discarded Picarro cross-cal '
                     'candidate and why tank wins at plume concentrations.')}
    if _anchor:
        # Recorded, not applied (CAL_BASELINE_ANCHOR_APPLY=False). `intercept` above is
        # the tank fit; `intercept_if_applied` is the alternative that matches Picarro's
        # ambient median. Slope is identical for both -- only the offset differs -- so
        # applying it downstream is: cal = (measured - intercept_if_applied) / slope.
        # r2/n_points describe the TANK fit deliberately: they are the span's
        # diagnostics, and the anchor has no r2 of its own (it is one quantile match).
        _rec['baseline_anchor'] = {
            'applied': bool(CAL_BASELINE_ANCHOR_APPLY),
            'kind': 'cross_instrument_harmonization',
            'reference': 'Picarro (tank-corrected)',
            'quantile': _anchor['anchor_q'],
            'intercept_tank_fit': _anchor['intercept_prior'],
            'intercept_if_applied': _anchor['intercept'],
            'z_ref': _anchor['z_ref'], 'z_tgt': _anchor['z_tgt'],
            'n_paired': _anchor['n_anchor'],
            'population': 'WYO co-deployment dates, tank windows excluded',
            'note': CAL_BASELINE_ANCHOR[('CH4', inst)],
            'why_not_applied': (
                'A *_cal column in this dataset means traceable to the certified ladder. '
                'This offset is harmonization to another instrument, so it is recorded '
                'rather than folded in. It is a pure constant: dCH4 is unaffected, and '
                'absolute inter-instrument agreement on the WYO days is one line away.'),
        }
    corrections.append(_rec)
for inst, c in RESULTS['C3H8']['applied'].items():
    if c is None:
        continue
    corrections.append({'gas': 'C3H8', 'instrument': inst, 'col_in': 'C3H8_ppm', 'col_out': 'C3H8_ppm' + COL_SUFFIX_TRACEABLE,
                        'scale_in': 1.0, 'method': 'tank_multipoint',
                        'traceability': 'certified_tank_ladder', 'confidence': 'high',
                        'cal_date': CAL_DATE_CANONICAL, 'slope': c['slope'], 'intercept': c['intercept'],
                        'r2': c['r2'], 'n_points': c['n'],
                        'method_selection_note': 'CAL_METHOD_LOCKED["C3H8"]="tank" -- forced, only Ultra321 has this channel.'})
# C2H6 emits *_xcal, never *_cal: there is no certified anchor spanning the plume range,
# so every one of these is a transfer onto Ultra460's scale. Ultra460 itself is included
# with the identity transform so all three instruments expose the same column.
for inst, c in RESULTS['C2H6']['applied'].items():
    col_in, scale_in = C2H6_INPUT[inst]
    _is_ref = bool(c.get('identity'))
    _rec = {'gas': 'C2H6', 'instrument': inst, 'col_in': col_in,
            'col_out': 'C2H6_ppb' + COL_SUFFIX_TRANSFERRED,
            'scale_in': scale_in,
            'method': 'identity_reference' if _is_ref else 'zero_span_vs_ultra460',
            'traceability': 'none_cross_instrument_transfer',
            'confidence': 'reference_uncalibrated' if _is_ref else 'low',
            'note': ('This instrument IS the C2H6 reference: the transform is the identity '
                     '(slope 1, intercept 0) and C2H6_ppb_xcal equals the raw reading. It is '
                     'emitted so that all three C2H6 instruments carry the same column on the '
                     'same footing. Ultra460 C2H6 is NOT itself calibrated -- the tank certifies '
                     'only one point (NOAA 1.63 ppb), far below plume levels -- so the whole '
                     'C2H6 product is internally consistent but not traceable.'
                     if _is_ref else
                     'Baseline anchored to Ultra460 own ambient median (NOT the certified tank '
                     'zero) so the corrected reading matches Ultra460 by construction; span/gain '
                     'fit to plume peaks vs Ultra460. Ultra460 is the reference throughout, not '
                     'independently tank-validated over the plume range (only NOAA=1.63 ppb '
                     'certified). Because the anchor is ambient-matched rather than tank-zeroed, '
                     'this correction is NOT independently traceable to the certified zero.'),
            'method_selection_note': ('CAL_METHOD_LOCKED["C2H6"]="reference" -- forced, tank has '
                     'only 1 certified point far below plume levels; see '
                     '04_calibration_qc.ipynb assess_tank_coverage for the numeric case.'),
            'reference_instrument': 'Ultra460', 'n_peaks': c['n'],
            'baseline_method': 'ambient_median',
            'baseline_ref_ppb': c['z_ref'], 'baseline_ref_spread_ppb': c['z_ref_std'],
            'baseline_target_ppb': c['z_tgt'], 'baseline_target_spread_ppb': c['z_tgt_std'],
            'gain': c['gain'], 'slope': c['slope'], 'intercept': c['intercept'], 'r2': c['r2']}
    corrections.append(_rec)

# Attach any documented reliability caveat (config cell) to its correction record.
# Generic over species, so adding one is a config edit rather than a per-gas code change.
for c in corrections:
    _cav = CAL_CAVEATS.get((c['gas'], c['instrument']))
    if _cav:
        c['caveat'] = _cav

coefs_out = {
    'metadata': {
        'generated_utc': datetime.now(timezone.utc).isoformat(),
        'git_hash': git_hash, 'git_dirty': git_dirty,
        'upstream': {
            'wyo': upstream_ref(STAGE_03_DIR / 'apply_manifest_wyo.json'),
            'mml': upstream_ref(STAGE_03_DIR / 'apply_manifest_mml.json'),
        },
        'formula': 'calibrated = (measured * scale_in - intercept) / slope',
        'cal_date_canonical': CAL_DATE_CANONICAL,
        'cal_method_locked': CAL_METHOD_LOCKED,
        'method_selection_evidence': '04_calibration_qc.ipynb',
        'column_contract': {
            COL_SUFFIX_TRACEABLE: ('TRACEABLE -- derived against the certified tank/dilution ladder '
                                   'and nothing else. A value in this column is a measurement referred '
                                   'to a certified standard. Gases: CH4, C3H8.'),
            COL_SUFFIX_TRANSFERRED: ('TRANSFERRED -- harmonized to another instrument because no '
                                     'certified anchor spans the required range. Internally consistent '
                                     'across instruments, NOT traceable, not an absolute measurement. '
                                     'Gases: C2H6 (reference Ultra460).'),
            'per_correction_field': ("every correction carries a 'traceability' field: "
                                     "'certified_tank_ladder' or 'none_cross_instrument_transfer'."),
            'not_applied': ('Adjustments requiring a campaign-specific modelling choice are recorded '
                            'but NOT applied -- see any baseline_anchor block with "applied": false. '
                            'Those belong to downstream analysis, not to this ETL stage.'),
        },
        'baseline_anchor_applied': bool(CAL_BASELINE_ANCHOR_APPLY),
        'c2h6_method_note': ('C2H6 uses a zero+span cross-cal: ambient-baseline anchor (matches Ultra460 '
                             'ambient median, NOT the tank zero) + plume-peak gain vs Ultra460. Ultra460 '
                             'C2H6 is the reference and carries the identity transform; it is emitted as '
                             'C2H6_ppb_xcal so all three instruments share one column, and it is itself '
                             'uncalibrated.'),
    },
    'corrections': corrections,
}
STAGE_04_DIR.mkdir(parents=True, exist_ok=True)
coefs_path = STAGE_04_DIR / 'calibration_coefs.json'
with open(coefs_path, 'w') as f:
    json.dump(coefs_out, f, indent=2)
print(f'Saved {len(corrections)} corrections -> {coefs_path}')
_trace = sum(c['traceability'] == 'certified_tank_ladder' for c in corrections)
print(f'  {_trace} traceable ({COL_SUFFIX_TRACEABLE}), '
      f'{len(corrections) - _trace} transferred ({COL_SUFFIX_TRANSFERRED})')
if not CAL_BASELINE_ANCHOR_APPLY and CH4_ANCHOR_COEFS:
    print(f'  {len(CH4_ANCHOR_COEFS)} baseline anchor(s) recorded with "applied": false')

In [ ]:
pd.DataFrame(corrections)[['gas', 'instrument', 'col_in', 'col_out',
                            'method', 'confidence', 'slope', 'intercept', 'r2']].round(4)

---
## F — Apply calibration → `04_calibrated/`

`apply_calibration_to_dir` reads every good Stage 03 file (`Raw`+`Eng`; `bad/` and
`bad_timestamp/` are not descended into), adds the relevant `*_cal` columns and a
`cal_coefs_ref` column, and writes calibrated Parquet mirroring the Stage 03 layout.
Safe to re-run.

In [ ]:
CORR_BY_INST = {}
for c in corrections:
    CORR_BY_INST.setdefault(c['instrument'], {})[c['gas']] = c

APPLY_SUBDIRS = {
    'Picarro':  ('WYO_picarro', ['']),
    'Ultra460': ('WYO_aerisultra460', ['Raw', 'Eng']),
    'Ultra321': ('LANL_aerisultra321', ['Raw', 'Eng']),
    'Pico017':  ('LANL_aerispico017', ['Raw', 'Eng']),
}
apply_stats = {}
for inst, (inst_dir, subdirs) in APPLY_SUBDIRS.items():
    n_files = n_rows = 0
    for subdir in subdirs:
        src = STAGE_03_DIR / inst_dir / subdir if subdir else STAGE_03_DIR / inst_dir
        dst = STAGE_04_DIR / inst_dir / subdir if subdir else STAGE_04_DIR / inst_dir
        nf, nr = cal.apply_calibration_to_dir(src, dst, CORR_BY_INST.get(inst, {}))
        n_files += nf; n_rows += nr
    apply_stats[inst] = {'files': n_files, 'rows': n_rows}
    print(f'{inst:10s} {n_files:>4} files  {n_rows:>10,} rows -> {STAGE_04_DIR / inst_dir}')

apply_manifest = {'stage': '04_apply_calibration', 'run_utc': datetime.now(timezone.utc).isoformat(),
                  'git_hash': git_hash, 'git_dirty': git_dirty, 'coefs_source': str(coefs_path),
                  'instruments': apply_stats}
with open(STAGE_04_DIR / 'apply_manifest.json', 'w') as f:
    json.dump(apply_manifest, f, indent=2)
print('\nApply complete.')

**Sanity check 1 — on-disk output.** Read one calibrated file back off disk and
overlay its raw `CH4_ppm` against the written `CH4_ppm_cal`. This verifies the *actual
files*, not just the in-memory math.

In [ ]:
sample = sorted((STAGE_04_DIR / 'WYO_aerisultra460' / 'Raw').glob('*.parquet'))
if sample:
    df = pd.read_parquet(sample[0])
    fig = cal.plot_raw_vs_corrected(df['CH4_ppm'], df['CH4_ppm_cal'],
            f'Ultra460 CH4 raw vs calibrated (read back from disk) — {sample[0].name}', 'CH4 (ppm)')
    fig.show()
else:
    print('No calibrated Ultra460 files found — run the apply cell above first.')

**Sanity check 2 — magnitude of every correction.** Campaign-wide mean/std of
`(corrected − raw)` per correction, in output units. Nothing here should be physically
absurd (CH4 shifts of a few tenths of a ppm, etc.). `raw / scale_in` converts the
in-memory series into `col_in` units before applying, so the C2H6/Ultra321 unit twist is
handled correctly.

In [ ]:
srcmap = {'CH4': RESULTS['CH4']['raw'], 'C3H8': RESULTS['C3H8']['raw'], 'C2H6': RESULTS['C2H6']['raw']}
rows = []
for c in corrections:
    raw = srcmap[c['gas']].get(c['instrument'])
    if raw is None:
        continue
    corrected = cal.apply_linear(raw / c['scale_in'], c)   # raw/scale_in -> col_in units
    delta = corrected - raw
    rows.append({'gas': c['gas'], 'instrument': c['instrument'], 'col_out': c['col_out'],
                 'mean_delta': float(delta.mean()), 'std_delta': float(delta.std())})
pd.DataFrame(rows).round(3)

**Sanity check 3 — did calibration actually help?** The *same window* as Section B
(`ZOOM_START`/`ZOOM_END`), at native resolution: raw vs calibrated for CH4 (all four
instruments) and C2H6 (the three C2H6 channels). Raw traces sit offset from one another;
after calibration they should collapse toward a common value (CH4), or the corrected
Pico017/Ultra321 traces should sit on the uncalibrated Ultra460 reference (C2H6) — that
convergence, on real plume structure rather than a smeared average, is the whole point
of calibration.

In [ ]:
cal_ch4  = {inst: cal.apply_linear(RESULTS['CH4']['raw'][inst], c)
            for inst, c in RESULTS['CH4']['applied'].items() if c}
cal_c2h6 = {inst: cal.apply_linear(RESULTS['C2H6']['raw'][inst], c)
            for inst, c in RESULTS['C2H6']['applied'].items()}
cal_c2h6['Ultra460'] = RESULTS['C2H6']['raw']['Ultra460']   # reference — passthrough, uncalibrated

fig = cal.plot_timeseries_panels(
    panels=[('CH4 raw — instruments offset', 'CH4 (ppm)', RESULTS['CH4']['raw']),
            ('CH4 calibrated — should converge', 'CH4_cal (ppm)', cal_ch4),
            ('C2H6 raw', 'C2H6 (ppb)', RESULTS['C2H6']['raw']),
            ('C2H6 calibrated — Pico017/Ultra321 should sit on Ultra460', 'C2H6_cal (ppb)', cal_c2h6)],
    colors=INST_COLORS, t0=ZOOM_START, t1=ZOOM_END,
    title=f'End-of-calibration check — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC')
fig.show()

---
## G — Passthrough: everything else, unchanged

Straight copies (no recomputation) of Stage 03 good data that calibration doesn't touch:
Spectra/Spectralite for the three Aeris instruments (no concentration columns), and
GPS/Anem/Sprinter/LGR in full (no tank or cross-cal coverage — LGR wasn't deployed until
Mar 10, after all three cal events). These get **no** `cal_coefs_ref` column, which is how
to tell them apart from calibrated files.

In [ ]:
passthrough_stats = {}
SPECTRA_SUBDIR = {
    'Ultra460': ('WYO_aerisultra460', 'Spectralite'),
    'Ultra321': ('LANL_aerisultra321', 'Spectra'),
    'Pico017':  ('LANL_aerispico017', 'Spectra'),
}
for inst, (inst_dir, subdir) in SPECTRA_SUBDIR.items():
    n = cal.copy_passthrough_dir(STAGE_03_DIR / inst_dir / subdir, STAGE_04_DIR / inst_dir / subdir)
    passthrough_stats[f'{inst_dir}/{subdir}'] = n
    print(f'{inst_dir}/{subdir:12s} {n:>4} files')

for inst_dir in ['LANL_GPS', 'LANL_Anem', 'WYO_sprinter', 'UOU_LGR']:
    n = cal.copy_passthrough_dir(STAGE_03_DIR / inst_dir, STAGE_04_DIR / inst_dir)
    passthrough_stats[inst_dir] = n
    print(f'{inst_dir:24s} {n:>4} files')

with open(STAGE_04_DIR / 'passthrough_manifest.json', 'w') as f:
    json.dump({'stage': '04_passthrough', 'run_utc': datetime.now(timezone.utc).isoformat(),
               'note': ('Straight copies of Stage 03 good data with no calibration applied — '
                        'either no concentration columns (Spectra) or no tank/cross-cal coverage '
                        '(GPS, Anem, Sprinter, LGR). No cal_coefs_ref column added.'),
               'copied': passthrough_stats}, f, indent=2)
print('\nPassthrough complete.')

---
## H — Reconciliation & open items

**File-count reconciliation.** Every good Stage 03 file for the calibrated instruments
should appear in Stage 04. Any `MISMATCH` flag means something was silently dropped.

In [ ]:
def count_parquet(d):
    d = Path(d)
    return len(list(d.glob('*.parquet'))) if d.exists() else 0

print('Calibrated instruments (Stage 03 -> Stage 04 direct-child file counts):')
for inst, (inst_dir, subdirs) in APPLY_SUBDIRS.items():
    for subdir in subdirs:
        s3 = STAGE_03_DIR / inst_dir / subdir if subdir else STAGE_03_DIR / inst_dir
        s4 = STAGE_04_DIR / inst_dir / subdir if subdir else STAGE_04_DIR / inst_dir
        a, b = count_parquet(s3), count_parquet(s4)
        label = f'{inst_dir}/{subdir or "."}'
        flag = '' if a == b else '   <-- MISMATCH'
        print(f'  {label:40s} 03={a:>4}  04={b:>4}{flag}')

with open(STAGE_04_DIR / 'apply_manifest.json') as f:
    print('\napply_manifest instruments:', json.load(f)['instruments'])
with open(STAGE_04_DIR / 'passthrough_manifest.json') as f:
    print('passthrough_manifest copied:', json.load(f)['copied'])

### Open items

> **The product contract, restated.** `*_cal` = referred to the certified tank ladder
> (CH4, C3H8). `*_xcal` = harmonized to Ultra460 (C2H6), internally consistent but not
> absolute. Nothing that needs a campaign-specific modelling choice is applied to either;
> such adjustments are recorded in `calibration_coefs.json` with `"applied": false` and
> belong to downstream analysis. Several items below exist *because* of that line, and
> say so.

- **Ultra321 C2H6 — retained with a documented caveat (2026-08-27).** Its peaks are
  unreliable: the span fit is far looser than Pico017's and the residual correlates
  strongly with C3H8 at the matched peaks — spectral cross-talk no anchor choice fixes.
  It was dropped on 2026-08-26 for that reason and **reinstated on 2026-08-27**, because
  characterising *that* failure in the presence of propane/methane is a downstream
  deliverable that needs the calibrated column. Now listed in `CAL_CAVEATS` (config
  cell): applied and saved as `C2H6_ppb_xcal`, with a `caveat` field in its
  `calibration_coefs.json` record next to `confidence: "low"`. **Downstream must not use it as a quantitative C2H6
  measurement without propagating that uncertainty.** Live fit numbers are in Section D's
  fit output and QC notebook §E. **Ultra321 C3H8 and CH4 are unaffected.**
- **C2H6 baseline is anchored to Ultra460, not to the certified tank zero.** By design:
  Pico017/Ultra321's baseline is set to match Ultra460's own ambient median, not the
  tank's absolute zero (which would have left Pico017 ~5–6 ppb below Ultra460 — the two
  genuinely disagree on absolute zero). This makes the C2H6 product
  cross-instrument-consistent but **not independently traceable to the certified
  NOAA/N2-zero tank standards** — which is exactly what the `_xcal` suffix now says on
  the column itself, rather than leaving it to be discovered in a JSON `note`.
- **MML-date extrapolation — and positive evidence of drift there.** All three tank
  events fell inside the WYO window (Feb 3–12), so Feb-12 coefficients reach the Jan and
  March MML dates as an extrapolation with no direct tank evidence. It is now worse than
  unvalidated: Pico017 and Ultra321 are co-located on **every** MML date, and their
  tank-corrected ambient CH4 difference moves **0.63 ppm** across the campaign (+0.895 on
  Jan 20 → +0.540 Jan 22 → +0.280 Feb 4 → +0.264 Mar 10) while holding to 0.09 ppm inside
  the WYO week. At least one of the two drifted substantially; with only two co-located
  instruments and no third reference (UOU_LGR exists on Mar 10 alone) **this data cannot
  say which.** ~18% of each LANL instrument's rows sit on MML sessions. Characterising and
  correcting that drift is post-analysis, not ETL.
- **⏳ Per-day residual offsets — why the CH4 baseline anchor is recorded but NOT applied.**
  Raised 2026-08-31 from looking at the Section B/F timeseries, and the decisive input to the
  2026-09-01 product decision. A single campaign-wide anchor does not make the traces
  overlap on every day, so applying it would buy uneven, hard-to-describe agreement at the
  cost of the one property `*_cal` is supposed to guarantee.

  Median offset vs tank-corrected Picarro, per WYO date, **as the anchor would leave them
  if applied** (ppm):

  | date | Ultra460 | Ultra321 | Pico017 |
  |---|---|---|---|
  | 20260203 | −0.0253 | +0.0956 | — |
  | 20260205 | −0.0238 | +0.0631 | +0.2891 |
  | 20260206 | −0.0571 | +0.0456 | +0.2814 |
  | 20260207 | +0.0054 | +0.0166 | +0.2854 |
  | 20260208 | +0.0051 | +0.0005 | +0.2852 |
  | 20260209 | +0.0056 | −0.0090 | +0.2835 |
  | 20260210 | +0.0111 | −0.0100 | +0.2662 |
  | 20260211 | +0.0100 | −0.0169 | +0.2753 |
  | 20260212 | +0.0104 | −0.0178 | +0.2836 |

  Three things are going on, and only the second is a defect:
  1. **Pico017 could never be anchored** (its baseline drifts — see the bullet below), so
     even with the anchor applied it would sit ~+0.28 ppm high on every day. Under the
     current decision no instrument is anchored, so this particular asymmetry is gone:
     all four are on the plain tank fit.
  2. **Ultra321's residual drifts monotonically** across the co-located week, +0.096 (Feb 3)
     → −0.018 (Feb 12); Ultra460 is flatter but still moves −0.057 → +0.011. A single
     campaign-median anchor cannot zero all nine days at once, so Feb 3/5/6 keep a visible
     residual while Feb 7–12 sit at ±0.01–0.02 ppm. **The campaign-aggregate held-out RMS
     (~0.03 ppm) averages this structure away — do not quote it as a per-day accuracy.**
  3. **The default `ZOOM_START` is Feb 5**, among the worst three hours in the campaign for
     this. The typical day looks considerably better.

  Reassuring, and worth not losing: within a window the paired residual **sd is only
  0.017–0.033 ppm**. The traces are parallel and tight — they track each other's *shape*
  closely and differ only in vertical offset. There is no span or timing problem here, and
  ΔCH4 is unaffected by any of it.

  **Options, and what was chosen (2026-09-01):**
  - **Ship the plain tank fit; record the anchor unapplied** — ✅ **the current choice.**
    Uniformly traceable, one rule for all four instruments, and the anchor is preserved in
    `baseline_anchor.intercept_if_applied` so nothing is lost. Anyone wanting absolute
    inter-instrument agreement on the WYO days applies a constant, in one line.
  - **Apply the campaign-wide anchor** — what the notebook did until 2026-09-01. Better
    absolute agreement on most WYO days, but it mixes a certified-ladder span with an
    instrument-referred offset inside a single `*_cal` column, and the per-day table above
    shows the agreement it buys is uneven anyway. Set `CAL_BASELINE_ANCHOR_APPLY = True`
    to restore it.
  - **Per-WYO-day anchoring** — zeroes every WYO day by construction. **Blocked on a
    structural change**: `calibration_coefs.json` and `cal.apply_calibration_to_dir` carry
    one coefficient set per `(gas, instrument)`, not per date. MML days have no Picarro, so
    they would need a documented fallback, making the product internally non-uniform. This
    is post-analysis work regardless.
- **CH4 ships on the plain tank fit for all four instruments; the baseline anchor is
  computed and recorded, not applied** (`CAL_BASELINE_ANCHOR_APPLY = False`). The
  alternative intercept for Ultra460 and Ultra321 — tank slope kept bit-for-bit, offset
  moved to match co-located Picarro's ambient median — is written to each record's
  `baseline_anchor` block with `"applied": false`, alongside `intercept_if_applied`.
  Applying it downstream is `cal = (measured - intercept_if_applied) / slope`.
  **Consequence worth noting: every CH4 instrument now gets identical treatment**, so the
  old Pico017-is-different asymmetry is gone from the shipped product. The cost is that
  absolute ambient CH4 still differs between instruments — Ultra460/Ultra321 by ~0.12–0.13
  ppm, Pico017 by ~0.28 ppm against Picarro — which downstream must account for when
  comparing *absolute* levels. **ΔCH4 is unaffected**, since every one of these is a
  constant.
- **C2H6 reference assumption.** The entire C2H6 calibration (baseline and span alike)
  rests on Ultra460's C2H6 being correct, which is assumed, not independently validated.
- **Method locks are a point-in-time decision.** `CAL_METHOD_LOCKED` reflects the
  comparison in `04_calibration_qc.ipynb` as of the date noted in that dict's comments.
  Re-run the QC notebook if new tank/ambient data should prompt revisiting a lock.